# gf_mul CNN classifier training

이 노트북은 기존 LDA classifier 대신 1D-CNN으로 `gf_mul` trace를 분류한다.

저장 구조:

```text
weight_cnn/
  cnn_op1_hw_{MODEL_INDEX}.pt
  cnn_out_hw_{MODEL_INDEX}.pt
  split_indices_{MODEL_INDEX}.npz
  cnn_train_summary_{MODEL_INDEX}.json
```

기본 target은 SASCA practical에 필요한 `op1_hw`, `out_hw`이다.

In [ ]:
import os, json, math, random
from pathlib import Path
import numpy as np
import h5py

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

print('torch:', torch.__version__)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE)

In [ ]:
# =====================
# User configuration
# =====================

# TODO: 네 trace h5 파일 경로로 바꿔줘.
TRACE_H5_PATH = '/Users/ijaeyeon/chipwhisperer/jupyter/user/batches_gf_mul_CW308_STM32F4/500000trace_1400point_seed12.h5'

# 모델 index. 새 모델을 만들 때마다 0,1,2,...처럼 바꾸면 됨.
MODEL_INDEX = 0

WEIGHT_DIR = Path('weight_cnn')
WEIGHT_DIR.mkdir(exist_ok=True)

TARGETS_TO_TRAIN = ['op1_hw', 'out_hw']

TEST_SIZE = 0.20
VAL_SIZE_FROM_TRAIN = 0.10
RANDOM_SEED = 1234

BATCH_SIZE = 256
EPOCHS = 30
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 7

# trace 구간 crop. None이면 전체 trace 사용.
CROP_START = None
CROP_END = None

USE_CLASS_WEIGHTS = True

In [ ]:
def gf_mul_ref_python(a, b):
    """GF(2^8) multiplication with primitive polynomial 0x11d.
    네 기존 gf_mul과 다르면 이 함수만 기존 함수로 교체해도 됨.
    """
    a = int(a) & 0xff
    b = int(b) & 0xff
    res = 0
    for _ in range(8):
        if b & 1:
            res ^= a
        carry = a & 0x80
        a = (a << 1) & 0xff
        if carry:
            a ^= 0x1d
        b >>= 1
    return res & 0xff

HW = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)

def _first_existing_dataset(h5, candidates):
    for name in candidates:
        if name in h5 and isinstance(h5[name], h5py.Dataset):
            return h5[name][:], name
    found = []
    def visit(name, obj):
        if isinstance(obj, h5py.Dataset):
            found.append(name)
    h5.visititems(visit)
    raise KeyError(f'Could not find any of {candidates}. Existing datasets: {found[:30]}')

def load_trace_h5(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'TRACE_H5_PATH not found: {path.resolve()}')
    with h5py.File(path, 'r') as f:
        traces, trace_key = _first_existing_dataset(f, ['traces', 'trace', 'X', 'samples', 'traces_p'])
        a, a_key = _first_existing_dataset(f, ['a', 'op0', 'operand0', 'input_a', 'labels/a'])
        b, b_key = _first_existing_dataset(f, ['b', 'op1', 'operand1', 'input_b', 'labels/b'])
        try:
            v, v_key = _first_existing_dataset(f, ['v', 'out', 'output', 'result', 'labels/v', 'labels/out'])
        except KeyError:
            v = np.array([gf_mul_ref_python(x, y) for x, y in zip(a, b)], dtype=np.uint8)
            v_key = 'computed gf_mul_ref_python(a,b)'
    traces = np.asarray(traces)
    if traces.ndim != 2:
        raise ValueError(f'traces must be 2D (N,T), got {traces.shape}')
    a = np.asarray(a, dtype=np.uint8).reshape(-1)
    b = np.asarray(b, dtype=np.uint8).reshape(-1)
    v = np.asarray(v, dtype=np.uint8).reshape(-1)
    n = min(len(traces), len(a), len(b), len(v))
    traces, a, b, v = traces[:n], a[:n], b[:n], v[:n]
    print('loaded:', path)
    print('trace key:', trace_key, traces.shape)
    print('a key:', a_key, 'b key:', b_key, 'v key:', v_key)
    print('N =', n)
    return traces, a, b, v

def make_labels(a, b, v):
    return {
        'op0_val': a.astype(np.int64),
        'op1_val': b.astype(np.int64),
        'out_val': v.astype(np.int64),
        'op0_hw': HW[a].astype(np.int64),
        'op1_hw': HW[b].astype(np.int64),
        'out_hw': HW[v].astype(np.int64),
    }

In [ ]:
traces, a, b, v = load_trace_h5(TRACE_H5_PATH)
labels = make_labels(a, b, v)

if CROP_START is not None or CROP_END is not None:
    traces = traces[:, CROP_START:CROP_END]
    print('cropped traces:', traces.shape)

traces = traces.astype(np.float32)
N, T = traces.shape
print('trace shape:', traces.shape)
print('targets:', TARGETS_TO_TRAIN)
for t in TARGETS_TO_TRAIN:
    minlength = 9 if t.endswith('_hw') else 256
    print(t, 'classes:', np.unique(labels[t]), 'hist first bins:', np.bincount(labels[t], minlength=minlength)[:12])

In [ ]:
# Split indices are shared across targets.
all_idx = np.arange(N)
trainval_idx, test_idx = train_test_split(all_idx, test_size=TEST_SIZE, random_state=RANDOM_SEED, shuffle=True)
train_idx, val_idx = train_test_split(trainval_idx, test_size=VAL_SIZE_FROM_TRAIN, random_state=RANDOM_SEED, shuffle=True)

# Normalize using train split only.
mean = traces[train_idx].mean(axis=0, keepdims=True)
std = traces[train_idx].std(axis=0, keepdims=True)
std[std < 1e-6] = 1.0

np.savez(WEIGHT_DIR / f'split_indices_{MODEL_INDEX}.npz', train_idx=train_idx, val_idx=val_idx, test_idx=test_idx)
print('split:', len(train_idx), len(val_idx), len(test_idx))
print('normalization:', mean.shape, std.shape)

In [ ]:
class TraceDataset(Dataset):
    def __init__(self, traces, y, indices, mean, std):
        self.traces = traces
        self.y = y.astype(np.int64)
        self.indices = np.asarray(indices)
        self.mean = mean.astype(np.float32)
        self.std = std.astype(np.float32)
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        idx = self.indices[i]
        x = (self.traces[idx:idx+1] - self.mean) / self.std  # shape (1,T)
        return torch.from_numpy(x.astype(np.float32)), torch.tensor(self.y[idx], dtype=torch.long)

class SimpleGFTraceCNN(nn.Module):
    def __init__(self, trace_len, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=11, padding=5),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=9, padding=4),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(16),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.25),
            nn.Linear(64 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

def evaluate(model, loader):
    model.eval()
    ys, preds, losses = [], [], []
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            losses.append(loss.item() * len(yb))
            preds.append(logits.argmax(dim=1).cpu().numpy())
            ys.append(yb.cpu().numpy())
    y = np.concatenate(ys)
    p = np.concatenate(preds)
    return np.sum(losses) / len(y), accuracy_score(y, p), y, p

In [ ]:
def train_one_target(target_name):
    y = labels[target_name]
    num_classes = 9 if target_name.endswith('_hw') else 256

    train_ds = TraceDataset(traces, y, train_idx, mean, std)
    val_ds = TraceDataset(traces, y, val_idx, mean, std)
    test_ds = TraceDataset(traces, y, test_idx, mean, std)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = SimpleGFTraceCNN(trace_len=T, num_classes=num_classes).to(DEVICE)

    if USE_CLASS_WEIGHTS:
        counts = np.bincount(y[train_idx], minlength=num_classes).astype(np.float32)
        weights = counts.sum() / np.maximum(counts, 1.0)
        weights = weights / weights.mean()
        criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE))
    else:
        criterion = nn.CrossEntropyLoss()

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', patience=3, factor=0.5)

    best_val_acc = -1.0
    best_state = None
    bad = 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        total = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            total_loss += loss.item() * len(yb)
            total += len(yb)
        val_loss, val_acc, _, _ = evaluate(model, val_loader)
        scheduler.step(val_acc)
        print(f'[{target_name}] epoch {epoch:02d}: train_loss={total_loss/total:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}')
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print('early stop')
                break

    model.load_state_dict(best_state)
    test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader)
    print(f'[{target_name}] TEST loss={test_loss:.4f} acc={test_acc:.4f}')
    print('confusion matrix:')
    print(confusion_matrix(y_true, y_pred, labels=np.arange(num_classes)))

    ckpt = {
        'model_state_dict': model.state_dict(),
        'target_name': target_name,
        'num_classes': num_classes,
        'trace_len': T,
        'crop_start': CROP_START,
        'crop_end': CROP_END,
        'mean': mean.astype(np.float32),
        'std': std.astype(np.float32),
        'model_index': MODEL_INDEX,
        'class_values': np.arange(num_classes, dtype=np.int64),
        'test_acc': float(test_acc),
        'val_acc': float(best_val_acc),
        'config': {
            'batch_size': BATCH_SIZE,
            'epochs': EPOCHS,
            'lr': LR,
            'weight_decay': WEIGHT_DECAY,
            'random_seed': RANDOM_SEED,
            'use_class_weights': USE_CLASS_WEIGHTS,
        }
    }
    out_path = WEIGHT_DIR / f'cnn_{target_name}_{MODEL_INDEX}.pt'
    torch.save(ckpt, out_path)
    print('saved:', out_path)
    return out_path, test_acc

results = {}
for target in TARGETS_TO_TRAIN:
    path, acc = train_one_target(target)
    results[target] = {'path': str(path), 'test_acc': float(acc)}

with open(WEIGHT_DIR / f'cnn_train_summary_{MODEL_INDEX}.json', 'w') as fp:
    json.dump(results, fp, indent=2)
print(results)